In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

In [46]:
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("cat_breeds_clean.csv", delimiter=";")

In [3]:
data

,Breed,Age_in_years,Age_in_months,Gender,Neutered_or_spayed,Body_length,Weight,Fur_colour_dominant,Fur_pattern,Eye_colour,Allowed_outdoor,Preferred_food,Owner_play_time_minutes,Sleep_time_hours,Country,Latitude,Longitude
0,Angora,0.25,3,female,False,19,2.0,white,solid,blue,False,wet,46,16,France,43.296482,5.369780
1,Angora,0.33,4,male,False,19,2.5,white,solid,blue,False,wet,48,16,France,43.611660,3.877710
2,Angora,0.50,6,male,False,20,2.8,black,solid,green,False,wet,41,11,France,44.837789,-0.579180
3,Angora,0.50,6,female,False,21,3.0,white,solid,blue,False,wet,24,8,France,43.611660,3.877710
4,Angora,0.50,6,male,False,21,3.0,red/cream,tabby,green,False,wet,51,10,France,48.864716,2.349014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066,Maine coon,0.17,2,female,False,15,1.2,white,solid,blue,False,wet,35,20,UK,51.507351,-0.127758
1067,Maine coon,0.17,2,female,False,17,1.0,black,bicolor,blue,False,wet,36,19,UK,51.507351,-0.127758
1068,Maine coon,0.17,2,male,False,14,0.7,red/cream,tabby,blue,False,wet,20,20,UK,51.507351,-0.127758
1069,Maine coon,0.17,2,male,False,16,1.1,red/cream,tabby,green,False,wet,34,19,UK,52.486244,-1.890401


In [4]:
data.describe()

,Age_in_years,Age_in_months,Body_length,Weight,Owner_play_time_minutes,Sleep_time_hours,Latitude,Longitude
count,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000
mean,4.845462,58.145658,44.003735,5.494613,23.049486,15.889823,44.439720,-60.178554
std,2.737469,32.849889,16.310308,2.292242,10.840922,2.621443,4.965876,45.364141
min,0.080000,1.000000,10.000000,0.500000,0.000000,8.000000,37.774930,-123.116226
25%,2.670000,32.000000,35.000000,3.900000,14.000000,14.000000,40.714270,-77.036370
50%,4.920000,59.000000,41.000000,5.000000,23.000000,16.000000,42.358430,-74.005970
75%,7.040000,84.500000,51.000000,7.075000,31.000000,18.000000,48.864716,-1.890401
max,11.250000,135.000000,102.000000,12.100000,60.000000,22.000000,53.800755,13.404954


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1071 entries, 0 to 1070
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Breed                    1071 non-null   str    
 1   Age_in_years             1071 non-null   float64
 2   Age_in_months            1071 non-null   int64  
 3   Gender                   1071 non-null   str    
 4   Neutered_or_spayed       1071 non-null   bool   
 5   Body_length              1071 non-null   int64  
 6   Weight                   1071 non-null   float64
 7   Fur_colour_dominant      1071 non-null   str    
 8   Fur_pattern              1071 non-null   str    
 9   Eye_colour               1071 non-null   str    
 10  Allowed_outdoor          1071 non-null   bool   
 11  Preferred_food           1071 non-null   str    
 12  Owner_play_time_minutes  1071 non-null   int64  
 13  Sleep_time_hours         1071 non-null   int64  
 14  Country                  1071 non-n

In [6]:
data.shape

(1071, 17)

In [7]:
data.isna().sum()

Breed                      0
Age_in_years               0
Age_in_months              0
Gender                     0
Neutered_or_spayed         0
Body_length                0
Weight                     0
Fur_colour_dominant        0
Fur_pattern                0
Eye_colour                 0
Allowed_outdoor            0
Preferred_food             0
Owner_play_time_minutes    0
Sleep_time_hours           0
Country                    0
Latitude                   0
Longitude                  0
dtype: int64

Датасет чистый, без пропусков и мусора

In [8]:
breed_dist = (
    data['Breed']
    .value_counts(dropna=False)
    .to_frame(name='count')
)
breed_dist['share'] = data['Breed'].value_counts(normalize=True, dropna=False)
breed_dist['percent'] = (breed_dist['share'] * 100).round(2)

In [9]:
breed_dist

,count,share,percent
Breed,,,
Ragdoll,435,0.406162,40.62
Maine coon,342,0.319328,31.93
Angora,294,0.274510,27.45


Баланс пород котиков

In [10]:
target_col = 'Breed'
X = data.drop(columns = [target_col])
Y = data[target_col]

In [11]:
cat_cols = [
    'Gender',
    'Neutered_or_spayed',
    'Fur_colour_dominant',
    'Fur_pattern',
    'Eye_colour',
    'Country',
    'Allowed_outdoor',
    'Preferred_food'
]

In [12]:
num_cols = [c for c in X.columns if c not in cat_cols]

In [13]:
preprocess = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

In [14]:
X_tf = preprocess.fit_transform(X)

In [15]:
X_tf

array([[ 1.      ,  0.      ,  1.      , ..., 16.      , 43.296482,
         5.36978 ],
       [ 0.      ,  1.      ,  1.      , ..., 16.      , 43.61166 ,
         3.87771 ],
       [ 0.      ,  1.      ,  1.      , ..., 11.      , 44.837789,
        -0.57918 ],
       ...,
       [ 0.      ,  1.      ,  1.      , ..., 20.      , 51.507351,
        -0.127758],
       [ 0.      ,  1.      ,  1.      , ..., 19.      , 52.486244,
        -1.890401],
       [ 1.      ,  0.      ,  1.      , ..., 20.      , 51.507351,
        -0.127758]], shape=(1071, 37))

In [16]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y  
)

In [17]:
model = Pipeline(steps=[
    ('preprocess', preprocess),           
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

In [18]:
model.fit(X_train, Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [19]:
model.score(X_test, Y_test)

0.7395348837209302

In [20]:
try:
    Y_pred = model.predict(X_test)
except Exception as e:
    print(repr(e))

In [21]:
report_dict = classification_report(Y_test, Y_pred, output_dict=True)
report = pd.DataFrame(report_dict).T

In [22]:
report

,precision,recall,f1-score,support
Angora,0.656716,0.745763,0.698413,59.000000
Maine coon,0.806452,0.724638,0.763359,69.000000
Ragdoll,0.755814,0.747126,0.751445,87.000000
accuracy,0.739535,0.739535,0.739535,0.739535
macro avg,0.739661,0.739176,0.737739,215.000000
weighted avg,0.744871,0.739535,0.740715,215.000000


In [23]:
results = []
metrics = [
    ('euclidean', {}),                  # L2
    ('manhattan', {}),                  # L1
    ('minkowski', {'p': 3}),           # p=3 как пример
]

for k in [1, 3, 5, 7, 9, 11]:
    for w in ['uniform', 'distance']:
        for metric, extra in metrics:
            model = Pipeline(steps=[
                ('preprocess', preprocess),
                ('knn', KNeighborsClassifier(
                    n_neighbors=k,
                    weights=w,
                    metric=metric,
                    **extra
                ))
            ])
            model.fit(X_train, Y_train)
            acc = accuracy_score(Y_test, Y_pred)
            results.append({
                'k': k,
                'weights': w,
                'metric': metric if metric != 'minkowski' else f'minkowski_p{extra.get("p", 2)}',
                'accuracy': acc
            })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)

In [24]:
results_df

,k,weights,metric,accuracy
0,1,uniform,euclidean,0.739535
1,1,uniform,manhattan,0.739535
20,7,uniform,minkowski_p3,0.739535
21,7,distance,euclidean,0.739535
22,7,distance,manhattan,0.739535
23,7,distance,minkowski_p3,0.739535
24,9,uniform,euclidean,0.739535
25,9,uniform,manhattan,0.739535
26,9,uniform,minkowski_p3,0.739535
27,9,distance,euclidean,0.739535


Теперь отмасштабируем числовые признаки:

In [25]:
preprocess_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', Pipeline(steps=[
            ('scaler', StandardScaler())
        ]), num_cols),
    ]
)

In [26]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)


Без масштабирования:

In [31]:
model_raw = Pipeline(steps=[
    ('preprocess', preprocess),            
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

С масштабированием:

In [28]:
model_scaled = Pipeline(steps=[
    ('preprocess', preprocess_scaled),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

In [32]:
model_raw.fit(X_train, Y_train)
model_scaled.fit(X_train, Y_train)

In [33]:
Y_pred_raw = model_raw.predict(X_test)
Y_pred_scaled = model_scaled.predict(X_test)

In [34]:
acc_raw = accuracy_score(Y_test, Y_pred_raw)
acc_scaled = accuracy_score(Y_test, Y_pred_scaled)

In [37]:
acc_raw = accuracy_score(Y_test, Y_pred_raw)
acc_scaled = accuracy_score(Y_test, Y_pred_scaled)

print('Точность без масштабирования:', acc_raw)
print('Точность с масштабированием:', acc_scaled)

Точность без масштабирования: 0.7395348837209302
Точность с масштабированием: 0.9209302325581395


С масштабированием модель стала заметно точнее.

In [47]:
base_model = Pipeline(steps=[
    ('preprocess', preprocess_scaled),     
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
}

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X, Y)

cv_results = pd.DataFrame(grid.cv_results_)

In [42]:
cv_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knn__metric,param_knn__n_neighbors,param_knn__weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009056,0.000901,0.005579,0.000403,euclidean,3,uniform,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.660465,0.957944,0.934579,0.948598,0.869159,0.874149,0.111267,20
1,0.007848,0.001692,0.004392,0.000738,euclidean,3,distance,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.665116,0.957944,0.934579,0.953271,0.869159,0.876014,0.110123,16
2,0.008112,0.002443,0.003667,0.000535,euclidean,5,uniform,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.665116,0.967290,0.957944,0.962617,0.883178,0.887229,0.115277,6
3,0.006494,0.002822,0.003391,0.000888,euclidean,5,distance,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.655814,0.962617,0.948598,0.962617,0.864486,0.878826,0.117346,12
4,0.006521,0.002810,0.004255,0.001770,euclidean,7,uniform,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.665116,0.967290,0.943925,0.962617,0.859813,0.879752,0.114117,10
5,0.006776,0.002151,0.003626,0.001417,euclidean,7,distance,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.669767,0.967290,0.939252,0.957944,0.850467,0.876944,0.111561,14
6,0.005007,0.001511,0.003341,0.000536,euclidean,9,uniform,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.646512,0.953271,0.953271,0.957944,0.855140,0.873228,0.119765,22
7,0.006496,0.002815,0.004037,0.001296,euclidean,9,distance,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.655814,0.957944,0.943925,0.962617,0.850467,0.874153,0.116574,18
8,0.007317,0.003055,0.003309,0.000212,euclidean,11,uniform,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.655814,0.953271,0.957944,0.953271,0.878505,0.879761,0.115822,8
9,0.007229,0.002785,0.003914,0.001707,euclidean,11,distance,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",0.651163,0.953271,0.953271,0.953271,0.841121,0.870419,0.117919,24


In [43]:
print('Лучшие параметры:', grid.best_params_)
print('Лучшая cv-accuracy:', grid.best_score_)

Лучшие параметры: {'knn__metric': 'manhattan', 'knn__n_neighbors': 7, 'knn__weights': 'distance'}
Лучшая cv-accuracy: 0.9114279504455552


С применением кросс-валидации точность получилась примерно такая же, как после масштабирования.